# 01 · Bronze checks
The `backfill_history` DAG loaded bronze month by month, for everything that **arrived** before go-live (2023-01-01). 
This notebook checks:
1. What bronze holds, per source
2. The same count taken straight from the raw files, without bronze's interval logic
3. Every record exactly once (no month boundary lost or doubled a record)
4. Raw totals against the proposal's EDA

In [1]:
import os
from datetime import datetime

import pandas as pd
import pyspark.sql.functions as F
import yaml

# run from repo root, where main.py, config/ and data/ are
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import main

with open("config/pipeline.yaml") as f:
    config = yaml.safe_load(f)
spark = main.get_spark(config)

go_live = datetime.fromisoformat(config["deployment"]["go_live"])
print("bronze holds everything that arrived before", go_live)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/05 10:22:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


bronze holds everything that arrived before 2023-01-01 00:00:00


## 1. What bronze holds

One row per source: rows, the number of runs that wrote them, and the first and last arrival time.

In [2]:
sources = ["crashes", "weather", "tracker", "taxi", "licences"]

rows = []
for source in sources:
    df = spark.read.parquet(f"datamart/bronze/{source}")
    stats = df.agg(
        F.count("*").alias("bronze_rows"),
        F.countDistinct("run_id").alias("runs"),
        F.min("_arrived_at").alias("first_arrival"),
        F.max("_arrived_at").alias("last_arrival"),
    ).collect()[0]
    rows.append({"source": source, **stats.asDict()})

bronze = pd.DataFrame(rows).set_index("source")
bronze

,bronze_rows,runs,first_arrival,last_arrival
source,,,,
crashes,527518,58,2018-03-01 00:06:00,2022-12-31 23:52:00
weather,62643,58,2018-02-28 18:00:00,2022-12-31 23:59:00
tracker,236525765,58,2018-03-01 00:10:00,2022-12-31 23:50:35
taxi,48176932,58,2018-03-01 00:00:00,2022-12-31 23:45:00
licences,27305,58,2013-09-23 00:00:00,2022-12-30 00:00:00


## 2. The same count, straight from the raw files
Read every raw file, take the column that says when a record arrives, and count the records that arrived before go-live. This deliberately does **not** reuse `utils/bronze.py`: an independent count is what makes it a check. The files are parsed the correct way (`multiLine=True`, `escape='"'`), because the licences file has line breaks inside its `location` field; with Spark's default parser, its 30,185 records read as 89,593 rows.

In [3]:
raw_arrival_column = {
    "crashes": "date_police_notified",
    "weather": "DATE",               # UTC
    "tracker": "time",
    "taxi": "trip_end_timestamp",
    "licences": "date_issued",
}
go_live_ts = F.lit(go_live).cast("timestamp_ntz")

rows = []
for source, column in raw_arrival_column.items():
    path = "data/" + config["sources"][source]["file"].replace("{month}", "*")
    arrived = F.col(column)
    if source == "weather":
        arrived = F.from_utc_timestamp(F.to_timestamp(arrived), "America/Chicago")
    arrived = arrived.cast("timestamp_ntz")

    stats = spark.read.csv(path, header=True, multiLine=True, escape='"').agg(
        F.count("*").alias("raw_rows"),
        F.sum((arrived < go_live_ts).cast("int")).alias("arrived_before_go_live"),
        F.sum((arrived >= go_live_ts).cast("int")).alias("arrived_from_go_live"),
    ).collect()[0]
    rows.append({"source": source, **stats.asDict()})

raw = pd.DataFrame(rows).set_index("source")
raw

,raw_rows,arrived_before_go_live,arrived_from_go_live
source,,,
crashes,600878,527518,73360
weather,71355,62643,8712
tracker,271389818,236525765,34864053
taxi,52517086,48176970,4337659
licences,30185,27305,2880


In [4]:
check = raw.join(bronze[["bronze_rows"]])
check["missing_from_bronze"] = check["arrived_before_go_live"] - check["bronze_rows"]
check["no_arrival_time"] = check["raw_rows"] - check["arrived_before_go_live"] - check["arrived_from_go_live"]
check

,raw_rows,arrived_before_go_live,arrived_from_go_live,bronze_rows,missing_from_bronze,no_arrival_time
source,,,,,,
crashes,600878,527518,73360,527518,0,0
weather,71355,62643,8712,62643,0,0
tracker,271389818,236525765,34864053,236525765,0,0
taxi,52517086,48176970,4337659,48176932,38,2457
licences,30185,27305,2880,27305,0,0


### Issues with taxi data
1. trips with **no end time** can't arrive at all; they are the `no_arrival_time` above;
2. trips whose end time is **before their start time** are source errors. If the end falls in the same month as the start, the trip still lands in bronze, with a meaningless arrival time, and silver drops it. If the end falls in an **earlier month**, no run reads the file the trip sits in (a month's run reads that month's file and the one before), so it never reaches bronze. Those are what `missing_from_bronze` counts.

In [5]:
taxi = spark.read.csv("data/chicago_taxi_trips/*.csv.gz", header=True, multiLine=True, escape='"')
start = F.col("trip_start_timestamp").cast("timestamp_ntz")
end = F.col("trip_end_timestamp").cast("timestamp_ntz")
ends_in_earlier_month = F.trunc(end, "month") < F.trunc(start, "month")

print("no end time:                        ", taxi.filter(end.isNull()).count())
print("ends before it starts:              ", taxi.filter(end < start).count())
print("... ends in an earlier month, before go-live:", taxi.filter(ends_in_earlier_month & (end < go_live_ts)).count())

no end time:                         2457
ends before it starts:               1572
... ends in an earlier month, before go-live: 38


## 3. Every record exactly once
Bronze runs are month-sized intervals of *arrival* time, and the monthly raw files are split by *start* time. A taxi trip that starts on 31 May and ends on 1 June sits in the May file but arrives in June. If the interval logic were wrong, such records would be lost (so `missing_from_bronze` above would be positive) or loaded twice (duplicated keys below).

In [7]:
keys = {
    "crashes": ["crash_record_id"],
    "weather": ["DATE_UTC", "REPORT_TYPE"],
    "tracker": ["record_id"],
    "taxi": ["trip_id"],
    "licences": ["id"],
}
for source, key in keys.items():
    df = spark.read.parquet(f"datamart/bronze/{source}")
    duplicated = df.groupBy(key).count().filter("count > 1").count()
    print(f"{source}: {duplicated} duplicated keys {key}")

crashes: 0 duplicated keys ['crash_record_id']
weather: 0 duplicated keys ['DATE_UTC', 'REPORT_TYPE']
tracker: 0 duplicated keys ['record_id']
taxi: 0 duplicated keys ['trip_id']
licences: 0 duplicated keys ['id']


## 4. Raw totals against the proposal EDA
The proposal's EDA (`slides/v3/charts/proposal_eda.ipynb`) counted the raw files with pandas. The raw totals here, counted by Spark, should agree.

In [8]:
eda = pd.Series({"crashes": 600_878, "weather": 71_355, "licences": 30_185}, name="proposal_eda")
totals = raw[["raw_rows"]].join(eda)
totals["raw_rows_millions"] = (totals["raw_rows"] / 1e6).round(2)
totals

,raw_rows,proposal_eda,raw_rows_millions
source,,,
crashes,600878,600878.0,0.60
weather,71355,71355.0,0.07
tracker,271389818,NaN,271.39
taxi,52517086,NaN,52.52
licences,30185,30185.0,0.03


In [9]:
# the EDA profiled one tracker month in detail: 2019-06 had 4.29M rows
tracker = spark.read.parquet("datamart/bronze/tracker")
print("tracker rows in run bulk_201906:", tracker.filter(F.col("run_id") == "bulk_201906").count())

tracker rows in run bulk_201906: 4285371


In [10]:
**Findings:**
- Bronze holds every raw record that arrived before go-live, exactly once, for crashes (527,518), weather (62,643), tracker (236,525,765) and licences (27,305): `missing_from_bronze` is 0, and no key is duplicated.
- Taxi has 48,176,932 trips in bronze. The only gaps are source errors: 2,457 trips with no end time, and 38 trips that end in an earlier month than they start (1,572 trips end before they start in total). Silver counts and drops them.
- The raw totals agree with the proposal EDA: crashes 600,878, weather 71,355, licences 30,185, taxi 52.52M, and tracker June 2019 4,285,371 (EDA: 4.29M). The licences match only when line breaks inside quoted fields are read correctly: Spark's default parser sees 89,593 rows.
- After go-live, 73,360 crashes, 8,712 weather reports, 34.9M tracker readings, 4.34M taxi trips and 2,880 licences remain. They arrive through the hourly feed in phase 2.

SyntaxError: invalid decimal literal (2640071599.py, line 4)